In [1]:
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

current_dir = os.path.dirname(os.path.abspath('__file__'))
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

from methods.sela.direct_methods import naive_gauss, gauss_partial_pivoting, gauss_scaled_pivoting, \
    gauss_complete_pivoting, LU_solve, LU_factoring # noqa: E402

from methods.sela.iterative_methods import jacobi, gauss_seidel, relaxing # noqa: E402

from utils.matrix import get_augmented_matrix # noqa: E402

# Métodos para solução de sistemas de equações lineares

Os métodos para resolver sistemas de equações lineares podem ser divididos em duas grandes categorias: métodos diretos e métodos iterativos.

Os métodos diretos resolvem o sistema em um número finito de passos e, em aritmética exata, fornecem a solução exata.

Os métodos iterativos geram uma sequência de aproximações que convergem para a solução exata, sendo especialmente úteis para sistemas grandes e esparsos.

__Características Comparativas__:
- Métodos Diretos: Mais precisos para sistemas pequenos ou médios; exigem mais memória; menos afetados pelo condicionamento do sistema.

- Métodos Iterativos: Mais eficientes para sistemas grandes; exigem menos memória; convergência depende do condicionamento do sistema; permitem controlar a precisão desejada.

## Solução de sistemas de eq. lineares

### Inicialização

In [2]:
# Matriz diagonalmente dominante - melhor para métodos iterativos
A = np.array([
    [10.0, -1.0, 2.0, 0.0],
    [-1.0, 11.0, -1.0, 3.0],
    [2.0, -1.0, 10.0, -1.0],
    [0.0, 3.0, -1.0, 8.0]
], dtype=float)

b = np.array([6.0, 25.0, -11.0, 15.0], dtype=float)
augmented_matrix = get_augmented_matrix(A, b)
tol = 1e-3
max_iter = 1000
relax_factor = 1.12  # Fator de relaxação para o método de relaxação

### Condicionamento

O condicionamento de um sistema linear pode ser verificado através de três métodos principais:

- a) Pequenas mudanças nos coeficientes da matriz A e no vetor b, e observar a mudança na solução x.

    Para verificar a condição de um sistema linear, podemos fazer pequenas perturbações nos coeficientes da matriz A e no vetor b, e observar como isso afeta a solução x. Se pequenas mudanças em A ou b resultarem em grandes mudanças em x, o sistema é considerado mal condicionado.

- b) Determinante da matriz A

    O determinante de uma matriz é uma medida de quão bem condicionada ela é. Se o determinante for próximo de zero, a matriz é considerada mal condicionada. Isso significa que pequenas mudanças nos dados de entrada podem levar a grandes mudanças na solução do sistema.

- c) Número de condição da matriz A

    O número de condição de uma matriz é uma medida de quão sensível a solução do sistema é a pequenas mudanças nos dados de entrada. Um número de condição alto indica que o sistema é mal condicionado, enquanto um número de condição baixo indica que o sistema é bem condicionado. O número de condição pode ser calculado usando a norma da matriz e sua inversa.

In [3]:
determinant = np.linalg.det(A)
cond = np.linalg.cond(A, p=1)

print(f"Determinant: {determinant}")
print(f"Condition number: {cond}")

Determinant: 7394.999999999988
Condition number: 3.1372549019607843


### Convergência

Uma matriz é dita diagonalmente dominante se, para cada linha, o valor absoluto do elemento da diagonal é maior ou igual à soma dos valores absolutos dos outros elementos da linha. Isso garante que o método de Gauss-Seidel converja.

In [4]:
for i in range(A.shape[0]):
    sum = A[i, :].sum()
    sum = sum - A[i, i]

    if np.abs(A[i, i]) < np.abs(sum):
        print(f"Linha {i}: {A[i, i]} < {sum}")
        print("Matriz não é diagonalmente dominante")
        break


### Solução

In [5]:
naive_gauss_solution = naive_gauss(augmented_matrix.copy())
gauss_partial_pivoting_solution = gauss_partial_pivoting(augmented_matrix.copy())
gauss_scaled_pivoting_solution = gauss_scaled_pivoting(augmented_matrix.copy())
gauss_complete_pivoting_solution = gauss_complete_pivoting(augmented_matrix.copy())
lu_solution = LU_solve(augmented_matrix.copy())

try:
    jacobi_solution = jacobi(augmented_matrix.copy(), tol, max_iter)
except Exception as e:
    print(f"Erro no método de Jacobi: {e}")
    jacobi_solution = None

try:
    gauss_seidel_solution = gauss_seidel(augmented_matrix.copy(), tol, max_iter)
except Exception as e:
    print(f"Erro no método de Gauss-Seidel: {e}")
    gauss_seidel_solution = None

try:
    relaxing_solution = relaxing(
        augmented_matrix.copy(), tol, max_iter, relax_factor
    )
except Exception as e:
    print(f"Erro no método de Relaxação: {e}")
    relaxing_solution = None


print("Solução dos métodos diretos:")
print(f"{'Naive Gauss:':<35} {naive_gauss_solution}")
print(f"{'Gauss com Pivoteamento Parcial:':<35} {gauss_partial_pivoting_solution}")
print(f"{'Gauss com Pivoteamento Escalonado:':<35} {gauss_scaled_pivoting_solution}")
print(f"{'Gauss com Pivoteamento Completo:':<35} {gauss_complete_pivoting_solution}")
print(f"{'Fatoração LU:':<35} {lu_solution}")

print("\nSolução dos métodos iterativos:")
print(f"{'Jacobi:':<35} {jacobi_solution}")
print(f"{'Gauss-Seidel:':<35} {gauss_seidel_solution}")
print(f"{'Relaxação:':<35} {relaxing_solution}")

print("\nComparação com a solução correta usando numpy:")
print(f"{'Solução correta (numpy):':<35} {np.linalg.solve(A, b)}")

Solução dos métodos diretos:
Naive Gauss:                        [ 1.  2. -1.  1.]
Gauss com Pivoteamento Parcial:     [ 1.  2. -1.  1.]
Gauss com Pivoteamento Escalonado:  [ 1.  2. -1.  1.]
Gauss com Pivoteamento Completo:    [ 1.  2. -1.  1.]
Fatoração LU:                       [ 1.  2. -1.  1.]

Solução dos métodos iterativos:
Jacobi:                             [ 0.99993981  1.99998904 -0.99997989  1.00000662]
Gauss-Seidel:                       [ 0.99993981  1.99998904 -0.99997989  1.00000662]
Relaxação:                          [ 0.99980459  1.99998661 -0.99995776  0.99996999]

Comparação com a solução correta usando numpy:
Solução correta (numpy):            [ 1.  2. -1.  1.]


## Calculo de inversa

### Inicialização

In [6]:
A = np.array([
    [4.0, 1.0, 0.0, 1.0],
    [1.0, 3.0, -1.0, 0.0],
    [0.0, -1.0, 5.0, 2.0],
    [1.0, 0.0, 2.0, 4.0]
], dtype=float)
EYE = np.eye(A.shape[0], dtype=np.float64)

### Verificar se é invertível

Uma matriz é não-invertível (ou singular) quando:

- O determinante é zero (det(A) = 0)
- Existe pelo menos uma linha ou coluna que é combinação linear das outras
- Durante o processo de redução por linha, pelo menos uma linha da matriz escalonada é composta apenas de zeros
- O sistema homogêneo AX = 0 admite infinitas soluções

In [7]:
L, U = LU_factoring(A)
print("Matriz U:")
print(U)

determinant = np.linalg.det(A)
print(f"Determinante da matriz A: {determinant:.7e}")

Matriz U:
[[ 4.          1.          0.          1.        ]
 [ 0.          2.75       -1.         -0.25      ]
 [ 0.          0.          4.63636364  1.90909091]
 [ 0.          0.          0.          2.94117647]]
Determinante da matriz A: 1.5000000e+02


### Calcular a inversa

In [8]:
A_inv = np.zeros_like(A, dtype=np.float64)

for i in range(A.shape[0]):
    tmp_A_augmented = np.hstack((A, EYE[:, i].reshape(-1, 1)))
    X = LU_solve(tmp_A_augmented)

    A_inv[:, i] = X
print("Matriz inversa de A:")
print(A_inv)
print("Verificação da inversa:")
print(np.allclose(A @ A_inv, EYE))
print(np.linalg.inv(A))
print("A @ A inversa:")
print(A @ A_inv)

Matriz inversa de A:
[[ 0.29333333 -0.09333333  0.01333333 -0.08      ]
 [-0.09333333  0.39333333  0.08666667 -0.02      ]
 [ 0.01333333  0.08666667  0.27333333 -0.14      ]
 [-0.08       -0.02       -0.14        0.34      ]]
Verificação da inversa:
True
[[ 0.29333333 -0.09333333  0.01333333 -0.08      ]
 [-0.09333333  0.39333333  0.08666667 -0.02      ]
 [ 0.01333333  0.08666667  0.27333333 -0.14      ]
 [-0.08       -0.02       -0.14        0.34      ]]
A @ A inversa:
[[ 1.00000000e+00  2.42861287e-17  0.00000000e+00  0.00000000e+00]
 [ 3.46944695e-17  1.00000000e+00 -5.55111512e-17  0.00000000e+00]
 [ 0.00000000e+00 -6.93889390e-18  1.00000000e+00  0.00000000e+00]
 [ 5.55111512e-17  1.38777878e-17  0.00000000e+00  1.00000000e+00]]
